# Street Median — Constrained Urban Greening

**Scenario:** A 0.15-hectare street median along a major boulevard in downtown Abu Dhabi. The median is 300m long × 5m wide, surrounded by asphalt on both sides, with underground utilities limiting root depth to 1.5m.

**Climate:** Arid desert hot (Köppen BWh). Reflected heat from pavement adds 3-5°C to ambient temperature.

**Soil:** Imported sandy loam with moderate salinity (3.8 dS/m). Limited depth due to utilities.

**Design brief:** Maximum shade and cooling with minimum water use. Low maintenance. Species must tolerate reflected heat, limited root space, and occasional saline irrigation. Height restrictions apply due to overhead power lines (max 6m).

---

## 1. Define the Street Median Site

In [ ]:
from natureos.site import Site, SoilProfile, ClimateZone, LandUse, SoilTexture

median_site = Site(
    name="Corniche Boulevard Median",
    climate_zone=ClimateZone.BWH,
    soil=SoilProfile(
        texture=SoilTexture.SANDY_LOAM,
        salinity_dsm=3.8,
        organic_matter_pct=0.6,
        ph=7.9,
        depth_cm=150.0
    ),
    area_hectares=0.15,
    land_use=LandUse.STREETSCAPE,
    annual_rainfall_mm=90.0,
    max_summer_temp_c=50.0,
    latitude=24.47,
    longitude=54.37
)

print(f"Site: {median_site.name}")
print(f"Area: {median_site.area_hectares} ha ({median_site.area_hectares * 10000:.0f} m²)")
print(f"Dimensions: ~300m × 5m")
print(f"Soil depth: {median_site.soil.depth_cm} cm (limited by utilities)")
print(f"Max ambient + reflected heat: ~{median_site.max_summer_temp_c}°C")
print(f"Constraints: Max height 6m (power lines), root depth 1.5m (utilities)")

## 2. Filter Species for Constrained Urban Conditions

In [ ]:
from natureos.data.mena_species import ALL_SPECIES
from natureos.species import GrowthForm, WaterRegime, ThermalTolerance

# Streetscape constraints:
# - Max height 6m (exclude Date Palm at 20m)
# - Must tolerate EXTREME heat (reflected pavement)
# - Low or very low water requirement
# - Shrub or small tree growth form (limited root space)

street_species = [
    sp for sp in ALL_SPECIES
    if sp.mature_height_m is not None and sp.mature_height_m <= 6.0
    and sp.thermal_tolerance == ThermalTolerance.EXTREME
    and sp.water_regime in (WaterRegime.VERY_LOW, WaterRegime.LOW)
    and sp.growth_form in (GrowthForm.SHRUB, GrowthForm.TREE)
]

print(f"Species meeting streetscape constraints: {len(street_species)}")
for sp in street_species:
    print(f"  • {sp.display_name}")
    print(f"    Height: {sp.mature_height_m}m, Water: {sp.water_regime.value}, Heat: {sp.thermal_tolerance.value}")

## 3. Habitat Suitability with Hard Constraints

In [ ]:
from natureos.engines.habitat import HabitatSuitability

suitability = HabitatSuitability(median_site)
results = suitability.evaluate_all(street_species)

print("Species ranked for street median conditions:\n")
for i, r in enumerate(results, 1):
    print(f"{i}. {r.summary()}")
    print(f"   Thermal: {r.factor_scores['thermal_compatibility']:.0%} | "
          f"Water: {r.factor_scores['water_compatibility']:.0%} | "
          f"Salinity: {r.factor_scores['salinity_compatibility']:.0%}")

## 4. Water Budget — Minimizing Municipal Demand

In [ ]:
from natureos.engines.water import WaterBudget

selected_species = [r.species for r in results[:4]]

water = WaterBudget(site=median_site, irrigation_efficiency=0.9)  # Drip irrigation
water_result = water.calculate(selected_species)

print(water_result.summary())
print(f"\nPer m²: {water_result.net_irrigation_required_m3 / (median_site.area_hectares * 10000):.3f} m³/m²/year")
print(f"Municipal water cost (est.): AED {water_result.net_irrigation_required_m3 * 3.5:.0f}/year")

## 5. Urban Heat Mitigation — The Primary Goal

In [ ]:
from natureos.engines.urban_heat import UrbanHeatMitigation

# Dense linear planting for maximum shade
planting_counts = {
    selected_species[0]: 40,   # Shrubs at 2m spacing along 300m
    selected_species[1]: 35,
    selected_species[2]: 30,
    selected_species[3]: 25,
}

heat = UrbanHeatMitigation(
    species_counts=planting_counts,
    site_area_m2=median_site.area_hectares * 10000
)
heat_result = heat.assess()

print(heat_result.summary())
print(f"\nSurface temperature reduction beneath canopy: {heat_result.avg_surface_cooling_c:.1f}°C")
print(f"This is critical for pedestrian comfort along the corridor.")
print(f"\nEquivalent cooling: {heat_result.cooling_capacity_kw:.0f} kW")
print(f"This offsets approximately {heat_result.cooling_capacity_kw / 1.5:.0f} typical car AC units running at full capacity.")

## 6. Biodiversity in Constrained Spaces

In [ ]:
from natureos.engines.biodiversity import BiodiversityIndex

bio = BiodiversityIndex(species_abundances=planting_counts)
bio_result = bio.calculate()

print(bio_result.summary())
print(f"\nFor a constrained streetscape, {bio_result.species_count} species with")
print(f"Shannon H' = {bio_result.shannon_index:.2f} represents good diversity.")
print(f"Native ratio: {bio_result.native_ratio:.0%}")

## 7. Design Brief — Formal Constraints

In [ ]:
from natureos.design_brief import (
    DesignBrief, Objective, ObjectiveType,
    Constraint, ConstraintType, MaintenanceLevel
)

brief = DesignBrief(
    name="Corniche Boulevard Median Planting",
    site=median_site,
    objectives=[
        Objective(objective_type=ObjectiveType.MAXIMIZE_SHADE, weight=0.35),
        Objective(objective_type=ObjectiveType.MINIMIZE_WATER_USE, weight=0.30),
        Objective(objective_type=ObjectiveType.MAXIMIZE_BIODIVERSITY, weight=0.20),
        Objective(objective_type=ObjectiveType.MINIMIZE_MAINTENANCE, weight=0.15),
    ],
    constraints=[
        Constraint(
            constraint_type=ConstraintType.MAX_HEIGHT,
            value=6.0,
            description="Overhead power lines — maximum mature height 6m"
        ),
        Constraint(
            constraint_type=ConstraintType.WATER_BUDGET,
            value=800.0,
            description="Maximum annual irrigation: 800 m³/ha"
        ),
    ],
    water_budget_m3_yr=120.0,
    maintenance_level=MaintenanceLevel.LOW,
    regulatory_standards=["Abu Dhabi UPC Streetscape Manual"],
    stakeholder_notes="Primary concern is reflected heat. Municipality wants visible greening with minimum water. Overhead lines restrict height."
)

print(brief.summary())

---

## Summary: Street Median

This notebook demonstrates NatureOS for **highly constrained urban greening**:

- **Physical constraints** — height limits (power lines), root depth (utilities), narrow width
- **Extreme microclimate** — reflected pavement heat adds 3-5°C
- **Primary objective: cooling** — shade and evapotranspiration are the design drivers
- **DesignBrief** — formal objectives and constraints model real-world project requirements

This is the hardest design scenario — maximum performance demanded from minimum space and resources. NatureOS handles it with the same engines, just different inputs.

---

*NatureOS Core v0.1.0 | MENA Species Dataset v0.1.0 | Apache 2.0 License*